# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Total image count

In [ ]:
import os

CHARTS_DIR = '/content/drive/MyDrive/DMIF/charts_final'
total = 0
candlestick = 0
heatmap = 0

for company in os.listdir(CHARTS_DIR):
    company_dir = os.path.join(CHARTS_DIR, company)
    if not os.path.isdir(company_dir):
        continue
    for f in os.listdir(company_dir):
        if f.endswith('.png'):
            total += 1
            if 'candlestick' in f:
                candlestick += 1
            elif 'heatmap' in f:
                heatmap += 1

print(f"Total images     : {total}")
print(f"Candlestick      : {candlestick}")
print(f"Heatmap          : {heatmap}")
print(f"Matched pairs    : {min(candlestick, heatmap)}")

Total images     : 111104
Candlestick      : 46766
Heatmap          : 64338
Matched pairs    : 46766


# Rebuild chart index

In [ ]:
import pandas as pd

records = []
for company in sorted(os.listdir(CHARTS_DIR)):
    company_dir = os.path.join(CHARTS_DIR, company)
    if not os.path.isdir(company_dir):
        continue
    for fname in sorted(os.listdir(company_dir)):
        if not fname.endswith('.png'):
            continue
        parts      = fname.replace('.png', '').split('_')
        label_str  = parts[-1]
        label      = 1 if label_str == 'UP' else 0
        chart_type = 'candlestick' if 'candlestick' in fname else 'heatmap'
        records.append({
            'path'      : os.path.join(company_dir, fname),
            'label'     : label,
            'company'   : company,
            'chart_type': chart_type,
            'filename'  : fname
        })

index_df = pd.DataFrame(records)
index_df.to_csv('/content/drive/MyDrive/DMIF/chart_index_final.csv', index=False)

print(f"Total indexed    : {len(index_df)}")
print(f"Candlestick      : {(index_df['chart_type']=='candlestick').sum()}")
print(f"Heatmap          : {(index_df['chart_type']=='heatmap').sum()}")
print(f"UP labels        : {(index_df['label']==1).sum()}")
print(f"DOWN labels      : {(index_df['label']==0).sum()}")
up_pct   = (index_df['label']==1).sum() / len(index_df) * 100
down_pct = (index_df['label']==0).sum() / len(index_df) * 100
print(f"Class balance    : UP {up_pct:.1f}% / DOWN {down_pct:.1f}%")

Total indexed    : 111104
Candlestick      : 46766
Heatmap          : 64338
UP labels        : 38428
DOWN labels      : 72676
Class balance    : UP 34.6% / DOWN 65.4%


In [ ]:
!pip install mplfinance --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.5 MB/s eta 0:00:00


# Config (all 80 companies)

In [ ]:
import os

ACCOUNT_RANGE = (0, 80)  # scan ALL companies

CSV_PATH   = '/content/drive/MyDrive/DMIF/master_dataset_clean_80.csv'
OUTPUT_DIR = '/content/drive/MyDrive/DMIF/charts_final'
LOCAL_CSV  = '/content/master_dataset.csv'

SEQ_LEN  = 30
STRIDE   = 5
IMG_SIZE = 224

HEATMAP_FEATURES = [
    'Open', 'High', 'Low', 'Close', 'Share_Volume',
    'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower',
    'MA_24', 'MA_30', 'SAR', 'Fib_0618'
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Scanning all {ACCOUNT_RANGE[1]} companies")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

Scanning all 80 companies
OUTPUT_DIR: /content/drive/MyDrive/DMIF/charts_final


# Copy CSV to local

In [ ]:
import shutil
print("Copying CSV to local disk...")
shutil.copy(CSV_PATH, LOCAL_CSV)
print("Done.")

Copying CSV to local disk...
Done.


# Load data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(LOCAL_CSV, parse_dates=['Date'])
df = df.sort_values(['Company_Code', 'Date']).reset_index(drop=True)

all_companies = sorted(df['Company_Code'].unique())
start_idx, end_idx = ACCOUNT_RANGE
my_companies = all_companies[start_idx:end_idx]

print(f"Total companies : {len(all_companies)}")
print(f"Scanning        : {len(my_companies)} companies")

Total companies : 80
Scanning        : 80 companies


# Chart functions

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

def generate_candlestick(window_df, save_path):
    fig, (ax1, ax2) = plt.subplots(
        2, 1,
        figsize=(2.24, 2.24),
        gridspec_kw={'height_ratios': [3, 1]},
        dpi=100
    )
    fig.patch.set_facecolor('#0d0d0d')
    ax1.set_facecolor('#0d0d0d')
    ax2.set_facecolor('#0d0d0d')

    for i, (_, row) in enumerate(window_df.iterrows()):
        o, h, l, c = row['Open'], row['High'], row['Low'], row['Close']
        color = '#00b4b4' if c >= o else '#ff3333'
        ax1.plot([i, i], [l, h], color=color, linewidth=0.8)
        body_bottom = min(o, c)
        body_height = abs(c - o) if abs(c - o) > 0 else 0.001
        ax1.bar(i, body_height, bottom=body_bottom,
                color=color, width=0.6, linewidth=0)

    for i, (_, row) in enumerate(window_df.iterrows()):
        color = '#00b4b4' if row['Close'] >= row['Open'] else '#ff3333'
        ax2.bar(i, row['Share_Volume'], color=color, width=0.6,
                linewidth=0, alpha=0.7)

    for ax in [ax1, ax2]:
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    plt.tight_layout(pad=0)
    plt.savefig(save_path, dpi=100,
                bbox_inches='tight',
                facecolor='#0d0d0d',
                pad_inches=0)
    plt.close(fig)


def generate_heatmap(window_df, save_path):
    available  = [f for f in HEATMAP_FEATURES if f in window_df.columns]
    corr_data  = window_df[available].copy()
    corr_data  = corr_data.loc[:, corr_data.std() > 0]

    if corr_data.shape[1] < 2:
        fig, ax = plt.subplots(figsize=(IMG_SIZE/100, IMG_SIZE/100), dpi=100)
        ax.set_facecolor('#0d0d0d')
        fig.patch.set_facecolor('#0d0d0d')
        plt.savefig(save_path, dpi=100, bbox_inches='tight',
                    facecolor='#0d0d0d', pad_inches=0)
        plt.close(fig)
        return

    corr_matrix = corr_data.corr()
    fig, ax = plt.subplots(figsize=(IMG_SIZE/100, IMG_SIZE/100), dpi=100)
    fig.patch.set_facecolor('#0d0d0d')

    sns.heatmap(
        corr_matrix, ax=ax,
        cmap='RdYlGn',
        vmin=-1, vmax=1,
        annot=False,
        linewidths=0,
        cbar=False,
        square=True
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')

    plt.tight_layout(pad=0)
    plt.savefig(save_path, dpi=100,
                bbox_inches='tight',
                facecolor='#0d0d0d',
                pad_inches=0)
    plt.close(fig)

# Scan and generate missing only

In [ ]:
import gc

total_generated = 0
total_skipped   = 0
total_errors    = 0
companies_done  = 0
companies_with_missing = []

print(f"Scanning {len(my_companies)} companies for missing images...\n")

for company in my_companies:
    company_df = df[df['Company_Code'] == company].copy()
    company_df = company_df.sort_values('Date').reset_index(drop=True)

    if len(company_df) < SEQ_LEN + 1:
        print(f"  ⚠ {company}: not enough data — skipping")
        continue

    company_dir = os.path.join(OUTPUT_DIR, company)
    os.makedirs(company_dir, exist_ok=True)

    company_df['Target'] = (
        company_df['Close'].shift(-1) > company_df['Close']
    ).astype(float)

    max_start         = len(company_df) - SEQ_LEN - 1
    company_generated = 0
    company_skipped   = 0
    company_errors    = 0

    for start in range(0, max_start, STRIDE):
        end       = start + SEQ_LEN
        window_df = company_df.iloc[start:end].copy()
        label     = company_df.iloc[end]['Target']

        if pd.isna(label):
            continue

        label_str = 'UP' if label == 1.0 else 'DOWN'
        pred_date = company_df.iloc[end]['Date'].strftime('%Y%m%d')

        cs_filename = f"{company}_candlestick_{pred_date}_{label_str}.png"
        hm_filename = f"{company}_heatmap_{pred_date}_{label_str}.png"
        cs_path     = os.path.join(company_dir, cs_filename)
        hm_path     = os.path.join(company_dir, hm_filename)

        cs_exists = os.path.exists(cs_path)
        hm_exists = os.path.exists(hm_path)

        # Skip if both exist
        if cs_exists and hm_exists:
            company_skipped += 2
            total_skipped   += 2
            continue

        # Generate missing candlestick
        if not cs_exists:
            try:
                generate_candlestick(window_df, cs_path)
                total_generated   += 1
                company_generated += 1
            except Exception as e:
                total_errors   += 1
                company_errors += 1
                if company_errors <= 1:
                    print(f"  ✗ CS error [{company}]: {e}")

        # Generate missing heatmap
        if not hm_exists:
            try:
                generate_heatmap(window_df, hm_path)
                total_generated   += 1
                company_generated += 1
            except Exception as e:
                total_errors   += 1
                company_errors += 1
                if company_errors <= 1:
                    print(f"  ✗ HM error [{company}]: {e}")

    # Memory cleanup
    plt.close('all')
    gc.collect()

    companies_done += 1

    if company_generated > 0:
        companies_with_missing.append(company)
        print(f"  ✅ {company}: {company_generated} generated | "
              f"{company_skipped} skipped | "
              f"{company_errors} errors")
    else:
        print(f"  ✓  {company}: all exist ({company_skipped} skipped)")

print(f"\n{'='*55}")
print(f"Companies scanned        : {companies_done}")
print(f"Companies with missing   : {len(companies_with_missing)}")
print(f"Total generated          : {total_generated}")
print(f"Total skipped            : {total_skipped}")
print(f"Total errors             : {total_errors}")
print(f"{'='*55}")

Scanning 80 companies for missing images...

  ✓  AAF.N: all exist (1284 skipped)
  ✓  AAIC.N: all exist (1880 skipped)
  ✓  ABAN.N: all exist (1816 skipped)
  ✓  ABL.N: all exist (1114 skipped)
  ✓  ACAP.N: all exist (2780 skipped)
  ✓  ACME.N: all exist (2520 skipped)
  ✓  AFS.N: all exist (224 skipped)
  ✓  AGAL.N: all exist (2146 skipped)
  ✓  AGPL.N: all exist (226 skipped)
  ✓  AGST.N: all exist (1196 skipped)
  ✓  AGST.X: all exist (40 skipped)
  ✓  AHPL.N: all exist (1944 skipped)
  ✓  AHUN.N: all exist (2554 skipped)
  ✓  AINS.N: all exist (790 skipped)
  ✓  ALLI.N: all exist (1610 skipped)
  ✓  ALUM.N: all exist (1106 skipped)
  ✓  AMF.N: all exist (786 skipped)
  ✓  AMSL.N: all exist (1948 skipped)
  ✓  APLA.N: all exist (1848 skipped)
  ✓  ASCO.N: all exist (2504 skipped)
  ✓  ASHO.N: all exist (1112 skipped)
  ✓  ASIR.N: all exist (2796 skipped)
  ✓  ASIY.N: all exist (1232 skipped)
  ✓  ASPH.N: all exist (1134 skipped)
  ✓  ATL.N: all exist (1720 skipped)
  ✓  ATLL.N: all

# Final count verification

In [ ]:
total = 0
candlestick = 0
heatmap = 0

for company in sorted(os.listdir(OUTPUT_DIR)):
    company_dir = os.path.join(OUTPUT_DIR, company)
    if not os.path.isdir(company_dir):
        continue
    for f in os.listdir(company_dir):
        if f.endswith('.png'):
            total += 1
            if 'candlestick' in f:
                candlestick += 1
            elif 'heatmap' in f:
                heatmap += 1

print(f"Total images  : {total}")
print(f"Candlestick   : {candlestick}")
print(f"Heatmap       : {heatmap}")

if candlestick == heatmap:
    print(f"\n✅ Counts match — ready for training!")
else:
    print(f"\n⚠ Still {abs(candlestick - heatmap)} missing — rerun Cell 8")

Total images  : 128676
Candlestick   : 64338
Heatmap       : 64338

✅ Counts match — ready for training!


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/DMIF/master_dataset_clean_80.csv')
print(f"Total columns: {len(df.columns)}")
print(f"\nAll columns:")
print(df.columns.tolist())

# These are non-feature columns to exclude
exclude = ['Date', 'Company_Code', 'Target',
           'Trade_Volume', 'Turnover', 'Share_Volume']

feature_cols = [c for c in df.columns if c not in exclude]
print(f"\nPotential LSTM features: {len(feature_cols)}")
print(feature_cols)

Total columns: 84

All columns:
['Date', 'Open', 'High', 'Low', 'Close', 'Trade_Volume', 'Share_Volume', 'Turnover', 'MA_24', 'MA_30', 'MA_500', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper', 'BB_Lower', 'BB_Middle', 'BB_Width', 'SAR', 'SAR_Trend', 'Fib_0236', 'Fib_0382', 'Fib_05', 'Fib_0618', 'Fib_0786', 'Volume_MA_10', 'Volume_Ratio', 'Log_Volume', 'Daily_Return', 'Log_Return', 'HL_Range', 'OC_Range', 'Volatility_10', 'W_High', 'W_Low', 'W_Close', 'W_Volume', 'W_Turnover', 'W_Days', 'W_RSI', 'W_MACD', 'W_MA_10', 'W_MA_20', 'W_Return', 'W_Volatility', 'M_High', 'M_Low', 'M_Close', 'M_Volume', 'M_Turnover', 'M_Days', 'M_RSI', 'M_MACD', 'M_MA_6', 'M_MA_12', 'M_Return', 'M_Volatility', 'Q_High', 'Q_Low', 'Q_Close', 'Q_Volume', 'Q_Turnover', 'Q_Days', 'Q_RSI', 'Q_MACD', 'Q_MA_4', 'Q_MA_8', 'Q_Return', 'Q_Volatility', 'Y_High', 'Y_Low', 'Y_Close', 'Y_Volume', 'Y_Turnover', 'Y_Days', 'Y_RSI', 'Y_MACD', 'Y_MA_3', 'Y_MA_5', 'Y_Return', 'Y_Volatility', 'Company_Code', 'Target']

Potent

In [ ]:
LSTM_FEATURES = [
    'Open', 'High', 'Low', 'Close', 'Log_Volume',
    'Daily_Return', 'Log_Return', 'HL_Range', 'OC_Range',
    'MA_24', 'MA_30', 'MA_500',
    'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist',
    'BB_Upper', 'BB_Lower', 'BB_Middle', 'BB_Width', 'Volatility_10',
    'SAR', 'SAR_Trend',
    'Fib_0236', 'Fib_0382', 'Fib_05', 'Fib_0618', 'Fib_0786',
    'Volume_MA_10', 'Volume_Ratio'
]

print(f"Feature count: {len(LSTM_FEATURES)}")
missing = [f for f in LSTM_FEATURES if f not in df.columns]
print(f"Missing features: {missing}")
print("✓ All good" if len(missing) == 0 else "✗ Fix missing features")

Feature count: 30
Missing features: []
✓ All good
